# 01 — Data Loading & Feature Engineering

Load raw AMLSim CSVs, perform feature engineering, compute per-node graph features, 
and build the SAR alert labels. All outputs saved as parquet for downstream notebooks.

In [ ]:
import pandas as pd
import numpy as np
import os

# ── Pipeline integration ──────────────────────────────────────────
# When run via the web app, AML_* env vars control paths & params.
# When run standalone in Jupyter, sensible defaults apply.
_RUN_DIR = os.environ.get("AML_RUN_DIR", "")

DATA_PATH = os.environ.get("AML_DATA_PATH", os.path.join("..", "demodata"))
OUTPUT_PATH = os.path.join(_RUN_DIR, "data") if _RUN_DIR else "data"
_max_rows_str = os.environ.get("AML_MAX_ROWS", "0")
MAX_ROWS = int(_max_rows_str) if _max_rows_str and _max_rows_str != "0" else None

os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Data source:", os.path.abspath(DATA_PATH))
print("Output dir: ", os.path.abspath(OUTPUT_PATH))
if MAX_ROWS:
    print(f"Row limit:   {MAX_ROWS:,}")
else:
    print("Row limit:   None (full dataset)")

## 1. Load Raw CSVs

In [ ]:
transactions = pd.read_csv(os.path.join(DATA_PATH, "transactions.csv"))
party = pd.read_csv(os.path.join(DATA_PATH, "party.csv"))
alerts = pd.read_csv(os.path.join(DATA_PATH, "alert_transactions.csv"))

# Apply row limit if set (limit transactions; party & alerts stay full)
if MAX_ROWS and len(transactions) > MAX_ROWS:
    print(f"Sampling {MAX_ROWS:,} of {len(transactions):,} transactions")
    transactions = transactions.sample(n=MAX_ROWS, random_state=42)

print(f"Transactions: {len(transactions):,} rows")
print(f"Party:        {len(party):,} rows")
print(f"Alerts:       {len(alerts):,} rows")
transactions.head()

## 2. Transaction Feature Engineering

In [3]:
# Map tx_type string to numeric code
TX_TYPE_MAP = {
    "CASH_IN": 0, "CASH_OUT": 1, "DEBIT": 2,
    "PAYMENT": 3, "TRANSFER": 4, "DEPOSIT": 4,
}

def tx_type_to_code(tx_type_str):
    prefix = tx_type_str.split("-")[0]
    return TX_TYPE_MAP.get(prefix, 99)

transactions["tx_type"] = transactions["tx_type"].apply(tx_type_to_code)
transactions = transactions.rename(columns={"src": "source", "dst": "target"})
transactions = transactions[["source", "target", "tran_id", "tx_type", "base_amt", "tran_timestamp"]]

print("tx_type distribution:")
print(transactions["tx_type"].value_counts())
transactions.head()

tx_type distribution:
tx_type
4    430744
Name: count, dtype: int64


,source,target,tran_id,tx_type,base_amt,tran_timestamp
0,ee8986ee,3bdc1134,1,4,405.69,2020-01-01T00:00:00.000Z
1,c5dccde2,34cdefef,5,4,599.78,2020-01-01T00:00:00.000Z
2,c613c146,f73cf66f,6,4,385.70,2020-01-01T00:00:00.000Z
3,c5dccde2,34cdefef,7,4,498.69,2020-01-01T00:00:00.000Z
4,c5dccde2,7b808486,8,4,200.29,2020-01-01T00:00:00.000Z


## 3. Party Feature Engineering

In [4]:
PARTY_TYPE_MAP = {"Organization": 0, "Individual": 1}

party = party.rename(columns={"partyId": "id", "partyType": "type"})
party["type"] = party["type"].map(PARTY_TYPE_MAP).fillna(99).astype(int)

print(f"Nodes: {len(party):,}")
print("Party type distribution:")
print(party["type"].value_counts())
party.head()

Nodes: 7,500
Party type distribution:
type
0    3750
1    3750
Name: count, dtype: int64


,id,type
0,b800b2bf,0
1,c5dccde2,1
2,34cdefef,0
3,c1bfb464,0
4,c613c146,1


## 4. Build Alert Nodes (SAR Labels)

In [5]:
# Convert is_sar to int
alerts["is_sar"] = alerts["is_sar"].apply(lambda x: 1 if str(x).lower() == "true" else 0)

# Join alerts with transactions to find which nodes are involved in SAR alerts
alert_txns = transactions.merge(alerts[["tran_id", "is_sar"]], on="tran_id", how="inner")
alert_txns = alert_txns[alert_txns["is_sar"] == 1]

# Collect unique source and target nodes from SAR transactions
sar_sources = alert_txns[["source"]].rename(columns={"source": "id"})
sar_targets = alert_txns[["target"]].rename(columns={"target": "id"})
sar_nodes = pd.concat([sar_sources, sar_targets]).drop_duplicates(subset=["id"])
sar_nodes["is_sar"] = 1

# Left join to all nodes — non-SAR get 0
alert_nodes = party[["id"]].merge(sar_nodes, on="id", how="left")
alert_nodes["is_sar"] = alert_nodes["is_sar"].fillna(0).astype(int)

print(f"SAR nodes:     {alert_nodes['is_sar'].sum():,}")
print(f"Non-SAR nodes: {(alert_nodes['is_sar'] == 0).sum():,}")
print(f"Total nodes:   {len(alert_nodes):,}")

SAR nodes:     644
Non-SAR nodes: 6,856
Total nodes:   7,500


## 5. Compute Per-Node Graph Features

These features give GraphSAGE richer input beyond just party type.

In [6]:
# Outgoing (sent) features
sent = transactions.groupby("source").agg(
    out_degree=("target", "count"),
    total_amount_sent=("base_amt", "sum"),
    avg_amount_sent=("base_amt", "mean"),
    unique_counterparties_sent=("target", "nunique"),
).rename_axis("id")

# Incoming (received) features
received = transactions.groupby("target").agg(
    in_degree=("source", "count"),
    total_amount_received=("base_amt", "sum"),
    avg_amount_received=("base_amt", "mean"),
    unique_counterparties_received=("source", "nunique"),
).rename_axis("id")

# Merge all features
node_features = party.set_index("id")
node_features = node_features.join(sent, how="left")
node_features = node_features.join(received, how="left")
node_features = node_features.fillna(0)

# Add SAR labels
node_features = node_features.join(alert_nodes.set_index("id")[["is_sar"]], how="left")
node_features["is_sar"] = node_features["is_sar"].fillna(0).astype(int)
node_features = node_features.reset_index()

print(f"\nNode features shape: {node_features.shape}")
print(f"\nColumns: {list(node_features.columns)}")
node_features.describe()


Node features shape: (7500, 11)

Columns: ['id', 'type', 'out_degree', 'total_amount_sent', 'avg_amount_sent', 'unique_counterparties_sent', 'in_degree', 'total_amount_received', 'avg_amount_received', 'unique_counterparties_received', 'is_sar']


,type,out_degree,total_amount_sent,avg_amount_sent,unique_counterparties_sent,in_degree,total_amount_received,avg_amount_received,unique_counterparties_received,is_sar
count,7500.000000,7500.000000,7.500000e+03,7500.000000,7500.000000,7500.000000,7.500000e+03,7500.000000,7500.000000,7500.000000
mean,0.500000,57.432533,3.083747e+04,438.410166,8.946667,57.432533,3.083747e+04,549.388983,8.946667,0.085867
std,0.500033,2080.955391,1.109847e+06,390.970992,116.936929,1309.901058,6.995453e+05,210.446440,48.165216,0.280186
min,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000e+00,0.000000,0.000000,4.000000,2.183398e+03,479.906250,3.000000,0.000000
50%,0.500000,1.000000,8.074250e+02,473.570000,1.000000,6.000000,3.584730e+03,531.678231,4.000000,0.000000
75%,1.000000,3.000000,1.964922e+03,586.187500,3.000000,11.000000,5.897862e+03,592.249688,7.000000,0.000000
max,1.000000,164241.000000,8.757877e+07,8187.952667,6742.000000,79715.000000,4.253405e+07,7412.211250,2884.000000,1.000000


## 6. Build Edges Table (Deduplicated)

In [7]:
# Deduplicate edges for graph structure (keep unique source→target pairs)
edges = transactions[["source", "target"]].drop_duplicates()
print(f"Total transaction rows: {len(transactions):,}")
print(f"Unique directed edges:  {len(edges):,}")

Total transaction rows: 430,744
Unique directed edges:  67,100


## 7. Save All Processed Data

In [8]:
transactions.to_parquet(os.path.join(OUTPUT_PATH, "transactions_processed.parquet"), index=False)
node_features.to_parquet(os.path.join(OUTPUT_PATH, "node_features.parquet"), index=False)
alert_nodes.to_parquet(os.path.join(OUTPUT_PATH, "alert_nodes.parquet"), index=False)
edges.to_parquet(os.path.join(OUTPUT_PATH, "edges.parquet"), index=False)

print("Saved to data/:")
for f in os.listdir(OUTPUT_PATH):
    size = os.path.getsize(os.path.join(OUTPUT_PATH, f))
    print(f"  {f:40s} {size / 1024:.1f} KB")

Saved to data/:
  alert_nodes.parquet                      81.3 KB
  transactions_processed.parquet           4559.6 KB
  edges.parquet                            327.9 KB
  node_features.parquet                    316.9 KB


## 8. Summary

In [9]:
print("=" * 50)
print("DATA PIPELINE SUMMARY")
print("=" * 50)
print(f"Nodes:              {len(node_features):,}")
print(f"  - SAR:            {node_features['is_sar'].sum():,}")
print(f"  - Non-SAR:        {(node_features['is_sar'] == 0).sum():,}")
print(f"  - Organizations:  {(node_features['type'] == 0).sum():,}")
print(f"  - Individuals:    {(node_features['type'] == 1).sum():,}")
print(f"Edges (unique):     {len(edges):,}")
print(f"Transactions (raw): {len(transactions):,}")
print(f"Node features:      {node_features.shape[1] - 2} (excl. id and is_sar)")
print(f"\nFeature columns:")
for col in node_features.columns:
    if col not in ["id", "is_sar"]:
        print(f"  {col:35s} mean={node_features[col].mean():.2f}  std={node_features[col].std():.2f}")

DATA PIPELINE SUMMARY
Nodes:              7,500
  - SAR:            644
  - Non-SAR:        6,856
  - Organizations:  3,750
  - Individuals:    3,750
Edges (unique):     67,100
Transactions (raw): 430,744
Node features:      9 (excl. id and is_sar)

Feature columns:
  type                                mean=0.50  std=0.50
  out_degree                          mean=57.43  std=2080.96
  total_amount_sent                   mean=30837.47  std=1109847.48
  avg_amount_sent                     mean=438.41  std=390.97
  unique_counterparties_sent          mean=8.95  std=116.94
  in_degree                           mean=57.43  std=1309.90
  total_amount_received               mean=30837.47  std=699545.28
  avg_amount_received                 mean=549.39  std=210.45
  unique_counterparties_received      mean=8.95  std=48.17
